In [1]:
import os
import glob
import pandas as pd
import shutil
from tqdm import tqdm

In [33]:
# Base directory containing subject folders
# base = r"D:\JHAhn\8641_conductivity\DTI_ALPS\2_Dementia\EPT2019"
# base = r"D:\JHAhn\8641_conductivity\DTI_ALPS\2_Dementia\EPT2020"
# base = r"D:\JHAhn\8641_conductivity\DTI_ALPS\2_Dementia\EPT2021"
# base = r"D:\JHAhn\8641_conductivity\DTI_ALPS\2_Dementia\EPT2022"
# base = r"D:\JHAhn\8641_conductivity\DTI_ALPS\2_Dementia\EPT2023"
# base = r"D:\JHAhn\8641_conductivity\DTI_ALPS\2_Dementia\EPT2024"
base = r"D:\JHAhn\8641_conductivity\DTI_ALPS\1_8641_CN_MCI"
out = r"D:\JHAhn\8641_conductivity\DTI_ALPS\DTI_ALPS_Result\2_Dementia"

In [34]:
## ALPS File making
# Patterns for subject folders
patterns = [
## 2019
    # os.path.join(base, 'MREPT2019AD*')
    # , os.path.join(base, 'ADEPT*')
## 2020
    # os.path.join(base, 'HFC2020AD*')
    # , os.path.join(base, 'MREPT2020AD*')
## 2021
    # os.path.join(base, 'HFC2021AD*')
## 2022
    # os.path.join(base, 'EPT2022AD*')
## 2023
    # os.path.join(base, 'EPT2023AD*')
## 2024
    # os.path.join(base, 'EPT2024AD*')
## CN & MCI
    os.path.join(base, 'MWI_CN*')
    , os.path.join(base, 'VSI*')
           ]

# Collect all matching folders
subject_folders = []
for pat in patterns:
    subject_folders.extend(sorted(glob.glob(pat)))

# Filenames & hemisphere/suffix mappings
file_info = [
    ('Auto_ALPS_index_LH.csv', 'LH', '-ALL'),
    ('Auto_ALPS_index_LH_maxbval800.csv', 'LH', '-b800'),
    ('Auto_ALPS_index_LH_maxbval2000.csv', 'LH', '-b2000'),
    ('Auto_ALPS_index_RH.csv', 'RH', '-ALL'),
    ('Auto_ALPS_index_RH_maxbval800.csv', 'RH', '-b800'),
    ('Auto_ALPS_index_RH_maxbval2000.csv', 'RH', '-b2000'),
]

# Original columns in each CSV files
orig_cols = ['Dxx_proj', 'Dyy_proj', 'Dzz_proj', 'Dxx_assoc', 'Dyy_assoc', 'Dzz_assoc', 'ALPS index']

# Build final columns' list
columns = ['index', 'PDI', 'PID', 'Year']
for _, hemi, suffix in file_info:
    for col in orig_cols:
        columns.append(f"{hemi}{suffix}_{col}")

# Initialise merged DataFrame
merged_df = pd.DataFrame(columns=columns)

# Iterate over each subject folder & extract data
for idx, folder in enumerate(subject_folders, start=1):
    # Extract year (e.g., 'EPT2019' -> '2019')
    parent_folder = os.path.basename(os.path.dirname(folder))
    year = parent_folder.replace('EPT', '')

    row_data = {'index': idx
                , 'PDI': os.path.basename(folder)
                , 'PID': ''
                , 'Year': year
               }

    stats_dir = os.path.join(folder, 'Results', 'Stats')

    # Read each CSV and map its values
    for filename, hemi, suffix in file_info:
        file_path = os.path.join(stats_dir, filename)
        if not os.path.exists(file_path):
            print(f"!!!! Warning: {file_path} not found. Skipping.")
            # Fill NaN for expected columns
            for col in orig_cols:
                row_data[f"{hemi}{suffix}_{col}"] = pd.NA
            continue

        df_temp = pd.read_csv(file_path)
        # Rename columns to include hemisphere and suffix
        rename_map = {col: f"{hemi}{suffix}_{col}" for col in orig_cols}
        df_temp.rename(columns=rename_map, inplace=True)

        # Extract values (assumes a single row per CSV)
        for new_col in rename_map.values():
            row_data[new_col] = df_temp.at[0, new_col]

    # Append the row to the merged DataFrame
    # merged_df = merged_df.append(row_data, ignore_index=True)
    merged_df.loc[idx-1] = row_data

merged_df

!!!! Warning: D:\JHAhn\8641_conductivity\DTI_ALPS\1_8641_CN_MCI\MWI_CN21_PYB_NODDI32\Results\Stats\Auto_ALPS_index_LH_maxbval800.csv not found. Skipping.
!!!! Warning: D:\JHAhn\8641_conductivity\DTI_ALPS\1_8641_CN_MCI\MWI_CN21_PYB_NODDI32\Results\Stats\Auto_ALPS_index_RH_maxbval800.csv not found. Skipping.
!!!! Warning: D:\JHAhn\8641_conductivity\DTI_ALPS\1_8641_CN_MCI\VSI41_8550_KDH_NODDI32\Results\Stats\Auto_ALPS_index_LH.csv not found. Skipping.
!!!! Warning: D:\JHAhn\8641_conductivity\DTI_ALPS\1_8641_CN_MCI\VSI41_8550_KDH_NODDI32\Results\Stats\Auto_ALPS_index_LH_maxbval800.csv not found. Skipping.
!!!! Warning: D:\JHAhn\8641_conductivity\DTI_ALPS\1_8641_CN_MCI\VSI41_8550_KDH_NODDI32\Results\Stats\Auto_ALPS_index_LH_maxbval2000.csv not found. Skipping.
!!!! Warning: D:\JHAhn\8641_conductivity\DTI_ALPS\1_8641_CN_MCI\VSI41_8550_KDH_NODDI32\Results\Stats\Auto_ALPS_index_RH.csv not found. Skipping.
!!!! Warning: D:\JHAhn\8641_conductivity\DTI_ALPS\1_8641_CN_MCI\VSI41_8550_KDH_NODDI32\Re

,index,PDI,PID,Year,LH-ALL_Dxx_proj,LH-ALL_Dyy_proj,LH-ALL_Dzz_proj,LH-ALL_Dxx_assoc,LH-ALL_Dyy_assoc,LH-ALL_Dzz_assoc,...,RH-b800_Dyy_assoc,RH-b800_Dzz_assoc,RH-b800_ALPS index,RH-b2000_Dxx_proj,RH-b2000_Dyy_proj,RH-b2000_Dzz_proj,RH-b2000_Dxx_assoc,RH-b2000_Dyy_assoc,RH-b2000_Dzz_assoc,RH-b2000_ALPS index
0,1,MWI_CN01_KNH_NODDI32,,1_8641_CN_MCI,0.853582,0.731372,1.259900,1.086533,1.012292,0.845369,...,1.100942,0.777850,1.285058,0.530153,0.458794,0.897422,0.691640,0.658275,0.507262,1.264723
1,2,MWI_CN04_SSS_NODDI32,,1_8641_CN_MCI,0.721595,0.654347,1.212313,0.875281,1.239043,0.555101,...,1.158229,0.546558,1.558936,0.465566,0.309110,0.883277,0.581649,0.675830,0.360248,1.564507
2,3,MWI_CN05_LSS_NODDI32,,1_8641_CN_MCI,0.648850,0.534350,1.036836,0.781160,1.029860,0.513538,...,0.964159,0.522780,1.461470,0.417672,0.294318,0.829131,0.546362,0.642656,0.389061,1.410687
3,4,MWI_CN06_KDS_NODDI32,,1_8641_CN_MCI,0.776232,0.855603,0.955848,0.964658,0.958752,0.661455,...,0.930347,0.521188,1.413185,0.482395,0.414536,0.825959,0.616620,0.619220,0.372377,1.396616
4,5,MWI_CN07_KSB_NODDI32,,1_8641_CN_MCI,0.760503,0.692654,1.243302,0.877096,1.144913,0.597675,...,1.235381,0.608289,1.319060,0.527325,0.397003,0.931220,0.529175,0.732712,0.387569,1.346594
5,6,MWI_CN09_YKJ_NODDI32,,1_8641_CN_MCI,0.721305,0.585058,1.168966,0.807095,1.140908,0.511759,...,1.095211,0.584096,1.317951,0.542380,0.394802,0.979794,0.549094,0.687467,0.426458,1.329024
6,7,MWI_CN10_CJS_NODDI32,,1_8641_CN_MCI,0.766402,0.500861,1.051554,0.775147,0.896206,0.576431,...,1.005834,0.533251,1.445616,0.512246,0.366998,0.793137,0.573144,0.709497,0.416002,1.386194
7,8,MWI_CN12_LJK_NODDI32,,1_8641_CN_MCI,0.809517,0.506511,1.159348,1.022137,0.895856,0.680670,...,1.038751,0.607542,1.487791,0.532226,0.303901,0.739463,0.582983,0.691410,0.438068,1.503040
8,9,MWI_CN13_KH_NODDI32,,1_8641_CN_MCI,0.661128,0.740881,1.190856,0.836728,1.072988,0.530080,...,1.255691,0.531792,1.079911,0.471419,0.527062,1.076215,0.440779,0.855464,0.395012,0.989289
9,10,MWI_CN14_PYJ_NODDI32,,1_8641_CN_MCI,0.667318,0.716731,1.135098,0.797246,1.217310,0.580188,...,1.058661,0.511396,1.390524,0.427364,0.340209,0.988714,0.533452,0.681404,0.379724,1.334591


In [35]:
# Save the merged DataFrame to a new CSV
# merged_df.to_excel(os.path.join(out, 'Auto_ALPS_index_2019.xlsx'), index=False)
# merged_df.to_excel(os.path.join(out, 'Auto_ALPS_index_2020.xlsx'), index=False)
# merged_df.to_excel(os.path.join(out, 'Auto_ALPS_index_2021.xlsx'), index=False)
# merged_df.to_excel(os.path.join(out, 'Auto_ALPS_index_2022.xlsx'), index=False)
# merged_df.to_excel(os.path.join(out, 'Auto_ALPS_index_2023.xlsx'), index=False)
# merged_df.to_excel(os.path.join(out, 'Auto_ALPS_index_2024.xlsx'), index=False)
merged_df.to_excel(os.path.join(out, 'Auto_ALPS_index_CN-MCI.xlsx'), index=False)

In [36]:
## Duplicate 'Results' folders

# out1 = f"{out}\EPT2019"
# out1 = f"{out}\EPT2020"
# out1 = f"{out}\EPT2021"
# out1 = f"{out}\EPT2022"
# out1 = f"{out}\EPT2023"
# out1 = f"{out}\EPT2024"
out1 = f"{out}\CN-MCI"
pattern_prefixes = [os.path.basename(pat).replace('*','') for pat in patterns]

for _, row in tqdm(merged_df.iterrows(), total=len(merged_df), desc="Copying Results folders"):
    pdi = row['PDI']
    # PDI가 patterns의 prefix로 시작할 때만 복사
    if not any(pdi.startswith(pref) for pref in pattern_prefixes):
        continue

    src = os.path.join(base, pdi, 'Results')
    dst = os.path.join(out1, f"{pdi}_Results")
    try:
        shutil.copytree(src, dst)
        print(f"Copied: {src} -> {dst}")
    except FileNotFoundError:
        print(f"Warning: Source directory not found: {src}")
    except FileExistsError:
        print(f"Warning: Destination already exists: {dst}")

Copying Results folders:   2%|▉                                                         | 1/59 [00:43<42:29, 43.96s/it]

Copied: D:\JHAhn\8641_conductivity\DTI_ALPS\1_8641_CN_MCI\MWI_CN01_KNH_NODDI32\Results -> D:\JHAhn\8641_conductivity\DTI_ALPS\DTI_ALPS_Result\2_Dementia\CN-MCI\MWI_CN01_KNH_NODDI32_Results


Copying Results folders:   3%|█▉                                                        | 2/59 [01:37<47:14, 49.73s/it]

Copied: D:\JHAhn\8641_conductivity\DTI_ALPS\1_8641_CN_MCI\MWI_CN04_SSS_NODDI32\Results -> D:\JHAhn\8641_conductivity\DTI_ALPS\DTI_ALPS_Result\2_Dementia\CN-MCI\MWI_CN04_SSS_NODDI32_Results


Copying Results folders:   5%|██▉                                                       | 3/59 [02:33<48:49, 52.31s/it]

...

Copied: D:\JHAhn\8641_conductivity\DTI_ALPS\1_8641_CN_MCI\MWI_CN82_LMR_NODDI32\Results -> D:\JHAhn\8641_conductivity\DTI_ALPS\DTI_ALPS_Result\2_Dementia\CN-MCI\MWI_CN82_LMR_NODDI32_Results


Copying Results folders:  97%|███████████████████████████████████████████████████████  | 57/59 [59:30<01:57, 58.85s/it]

Copied: D:\JHAhn\8641_conductivity\DTI_ALPS\1_8641_CN_MCI\MWI_CN83_LYS_NODDI32\Results -> D:\JHAhn\8641_conductivity\DTI_ALPS\DTI_ALPS_Result\2_Dementia\CN-MCI\MWI_CN83_LYS_NODDI32_Results


Copying Results folders: 100%|███████████████████████████████████████████████████████| 59/59 [1:00:29<00:00, 61.52s/it]

Copied: D:\JHAhn\8641_conductivity\DTI_ALPS\1_8641_CN_MCI\MWI_CN84_LHS_NODDI32\Results -> D:\JHAhn\8641_conductivity\DTI_ALPS\DTI_ALPS_Result\2_Dementia\CN-MCI\MWI_CN84_LHS_NODDI32_Results
